# TFM: Exploración de patrones discursivos asociados a la depresión y el suicidio en redes sociales utilizando modelos de lenguaje

**GRUPO:** Micaela Beti, Gabriela Martín, Sílvia Bayés, Ignasi Barnadas

## Configuración del Entorno y Diccionario de Búsqueda

El alcance territorial de esta fase del estudio se centra en **[NOMBRE DEL PAÍS]**. Para la extracción de datos, se implementa una estrategia de filtrado geográfico nativo, utilizando el identificador espacial de la API (`place_country:[CÓDIGO_ISO_DEL_PAÍS]`). Esta decisión metodológica garantiza la procedencia de la muestra basándose en la asignación geográfica del servidor de origen, lo que mitiga los falsos positivos territoriales y evita la inestabilidad computacional asociada a las búsquedas por radio de coordenadas (`geocode`).

Siguiendo el reparto demográfico proporcional sobre el volumen total de peticiones asignadas al proyecto, se establece un objetivo de recolección ajustado a **[NÚMERO_TARGET] usuarios únicos** para esta región. Posteriormente, se extraerá un contexto longitudinal de 25 publicaciones por perfil.

El diccionario de palabras clave utiliza una versión flexibilizada y adaptada a la variante dialectal local (n-gramas de severidad media-alta). El objetivo en esta fase temprana es maximizar el *Recall* (exhaustividad) en la captación de usuarios semilla, delegando la depuración de ruido (falsos positivos, cuentas de noticias o institucionales) a los posteriores filtros de procesamiento semántico impulsados por Modelos de Lenguaje Grande (LLMs).

In [ ]:
# Instalación de librerías necesarias
!pip install pandas requests spacy scrubadub google-generativeai
!python -m spacy download es_core_news_md

import requests
import pandas as pd
import time
import json
import os
import google.generativeai as genai
import spacy
import scrubadub
import re
from datetime import datetime, timezone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.2/65.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 636.5/636.5 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.7 MB/s eta 0:00:00
  Attempting uninstall: textblob
    Found existing installation: textblob 0.19.0
    Uninstalling textblob-0.19.0:
      Successfully uninstalled textblob-0.19.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 21.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
# ==========================================
# CONFIGURACIÓN GENERAL - CATALÁN
# ==========================================

COUNTRY = "Euskera"
COUNTRY_CODE = "euskera"
LANG = "eu"
PLACE_COUNTRY = "ES"


TWITTER_API_KEY = '####'
GEMINI_API_KEY = '####'


# Target de recolección para nivel nacional
TARGET_USERS = 10000

# Límites temporales
START_YEAR = 2015
END_YEAR = 2026

# =========================================================
# KEYWORDS ESPECÍFICAS
# =========================================================

KEYWORDS = [

    # IDEAZIO SUIZIDA
    'hil nahi dut', 'hil egin nahi dut', 'bere buruaz beste egin',
    'bizitza kendu nahi diot neure buruari',
    'ez dut bizi nahi', 'bizitzeko gogorik ez',
    'ez existitzeko gogoa', 'desagertu nahi dut',
    'betiko desagertu', 'hobe ez banintz existituko',
    'esnatu ez banintz', 'lo egin eta ez esnatu',
    'hilda egongo banintz', 'ez dut jarraitu nahi',
    'ezin dut bizitzen jarraitu',
    'suizidatzean pentsatzen', 'pentsamendu suizidak',
    'ideazio suizida', 'bulkada suizida',
    'hiltzeko arrazoiak', 'nola suizidatu',
    'agur gutuna', 'denarekin amaitu nahi dut',
    'ezin dut gehiago', 'inork ez ninduke faltan botako',
    'nire bizitzak ez du baliorik',
    'desagertu nahiko nuke',

    # DEPRESIOA ETA ETSIPENA
    'depresio larria', 'depresioa dut',
    'deprimituta nago', 'hutsik sentitzen naiz',
    'suntsituta nago', 'hautsita nago',
    'nire bizitza gorroto dut',
    'neure burua gorroto dut',
    'inori ez zaio axola',
    'gero eta okerrago sentitzen naiz',
    'denak gainditzen nau',
    'ez dut ezer sentitzen',
    'ezerk ez du zentzurik',
    'ez diot zentzurik aurkitzen',
    'egunak daramatzat negarrez',
    'ohetik altxatu ezinik',
    'bizitzeaz nekatuta nago',
    'zama bat naiz',
    'barrutik hautsita nago',
    'hondoratzen ari naiz',
    'buruak suntsitzen nau',
    'nire bizitza kaka bat da',
    'barrutik hilda nago',
    'oso bakarrik sentitzen naiz',
    'indarrik gabe nago',
    'mentalki agortuta nago',

    # AUTOLESIOAK
    'autolesioa', 'autolesioak',
    'neure buruari min egin',
    'ebakitzen dut neure burua',
    'ebaki egin naiz',
    'min egin nahi diot neure buruari',
    'min egiten diot neure buruari',
    'ebakitzeak lasaitzen nau',
    'mina merezi dut',
    'neure burua zigortzen dut',

    # SUFRIMENDU PSIKOLOGIKOA
    'laguntza psikologikoa behar dut',
    'psikologoarengana joan behar dut',
    'terapia behar dut',
    'norbaitek entzutea behar dut',
    'ez dakit nola jarraitu',
    'ez dakit zer egin nirekin',
    'hondoratzen ari naiz',
    'kontrola galtzen ari naiz',
    'terapiak ez du funtzionatzen',
    'buruak ez du gelditzen',
    'emozionalki ezin dut gehiago',
    'mugan nago',
    'mentalki hautsita nago',
    'hemen jarraitu nahi ez dut',
    'nire bizitzan harrapatuta sentitzen naiz',
    'depresioak irabazten dit',
    'ez dut irteerarik ikusten',
    'ez dago irteerarik',
    'ez dut existitu nahi',
    'bizitzea astuna egiten zait',
    'egunero okerrago nago',
    'hondoa jotzen ari naiz',
    'ez dut jarraitzeko arrazoirik',
    'bizitzarekin amaitu nahi dut',

    # HASHTAGS
    '#osasunmentala', '#depresioa', '#tristura',
    '#mentalhealth', '#mentalhealthawareness',
    '#mentalillness', '#mentalhealthmatters',
    '#therapy', '#depressed', '#healing',
    '#recovery', '#sadness', '#pain'
]

# =========================================================
# DIRECTORIO DE SALIDA
# =========================================================
OUTPUT_DIR = f"datos_{COUNTRY_CODE}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Extracción de usuarios semilla

En esta fase, se inicializa la búsqueda retrospectiva de usuarios semilla en Twitter (X). Para garantizar la robustez de la extracción a nivel nacional, se ha abandonado el filtrado por coordenadas puras y se ha implementado una estrategia de filtrado geográfico nativo, adoptando las siguientes decisiones metodológicas:

* **Estrategia de Chunking (Lotes):** Para respetar el límite de caracteres por consulta de la API de búsqueda avanzada, el diccionario de palabras clave se divide en lotes reducidos.
* **Búsqueda Cronológica Inversa:** El bucle de iteración se inicia en el año actual (2026) y retrocede secuencialmente hasta 2015. Esta decisión garantiza la captura prioritaria de usuarios con actividad discursiva reciente.
* **Filtrado Geográfico de Servidor:** En lugar de validaciones léxicas o radios GPS inestables, se integra en la consulta el identificador espacial de la API (`place_country`), asegurando que la asignación territorial la realiza el propio motor de Twitter. Adicionalmente, se omite el filtro de idioma, permitiendo que el propio diccionario de *keywords* actúe como restrictor lingüístico natural para aligerar la petición.
* **Gestión de Peticiones y Control de Fallos:** Se incorpora un control estricto del código de estado HTTP (`status_code`) y una latencia forzada (`time.sleep=3`) entre peticiones para prevenir bloqueos por exceso de tráfico (*Rate Limits*) al buscar volúmenes altos.
* **Trazabilidad Temporal:** Se registra el `seed_year` (año del tweet original) para habilitar análisis longitudinales posteriores sobre variaciones temáticas en el tiempo (*Timeline Shift*).

In [ ]:
# =========================================================
# PASO 1. EXTRACCIÓN DE USUARIOS SEMILLA (MODO PLACE_COUNTRY)
# =========================================================

# V3 para evitar conflictos con archivos anteriores
ruta_paso1 = f"{OUTPUT_DIR}/01_{COUNTRY_CODE}_dataset_usuarios_semilla_V3.csv"

def extract_seed_users_country(target_users, start_year=START_YEAR, end_year=END_YEAR):
    if not TWITTER_API_KEY:
        raise ValueError("Falta TWITTER_API_KEY.")

    print(f"[*] Paso 1: Iniciando extracción (Modo Country ID) en {COUNTRY}. Objetivo: {target_users} usuarios.")

    url = "https://api.twitterapi.io/twitter/tweet/advanced_search"
    headers = {"x-api-key": TWITTER_API_KEY}

    # Lotes de 7 palabras
    chunk_size = 7
    keyword_chunks = [KEYWORDS[i:i + chunk_size] for i in range(0, len(KEYWORDS), chunk_size)]
    users_data = {}

    for year in range(end_year, start_year - 1, -1):
        if len(users_data) >= target_users:
            break

        print(f"  -> Buscando datos del año {year}...")

        dt_start = datetime(year, 1, 1, 0, 0, 0, tzinfo=timezone.utc)
        since_ts = int(dt_start.timestamp())

        dt_end = datetime(year, 12, 31, 23, 59, 59, tzinfo=timezone.utc)
        until_ts = int(dt_end.timestamp())

        for chunk_idx, chunk in enumerate(keyword_chunks, 1):
            if len(users_data) >= target_users:
                break

            query_terms = [f'"{kw}"' for kw in chunk]

            # BÚSQUEDA PURA: Keywords + Operador de País
            base_query = (
                f"({' OR '.join(query_terms)}) "
                f"place_country:{PLACE_COUNTRY}"
            )

            query = f"{base_query} since_time:{since_ts} until_time:{until_ts}"
            params = {"query": query}

            try:
                response = requests.get(url, headers=headers, params=params)

                if response.status_code == 200:
                    tweets = response.json().get("tweets", [])

                    for t in tweets:
                        author = t.get("author", {})
                        user_name = author.get("userName")

                        if user_name and user_name not in users_data:
                            # Diccionario limpio, extracción basada 100% en la API
                            users_data[user_name] = {
                                "userName": user_name,
                                "description": author.get("description", "Sin biografía"),
                                "seed_tweet": t.get("text"),
                                "seed_year": year,
                                "territorio": COUNTRY
                            }
                else:
                    print(f"    [!] Error de API en chunk {chunk_idx} ({response.status_code})")

                # Pausa de seguridad para no saturar el endpoint buscando 1000 usuarios
                time.sleep(3)

            except requests.exceptions.RequestException as e:
                print(f"    [!] Error de conexión en el año {year}, chunk {chunk_idx}: {e}")

    df_users = pd.DataFrame.from_dict(users_data, orient="index").reset_index(drop=True)
    print(f"[+] Paso 1 completado. Usuarios semilla consolidados: {len(df_users)}")
    return df_users

# Ejecución limpia saltando chequeos de archivos viejos
df_semilla = extract_seed_users_country(TARGET_USERS)

# Guardado
df_semilla.to_csv(ruta_paso1, index=False, encoding="utf-8-sig")
print(f"[+] Paso 1 guardado en: {ruta_paso1}")

# Muestra en pantalla
display(df_semilla.head(10))

[*] Paso 1: Iniciando extracción (Modo Country ID) en Euskera. Objetivo: 10000 usuarios.
  -> Buscando datos del año 2026...
  -> Buscando datos del año 2025...
  -> Buscando datos del año 2024...
  -> Buscando datos del año 2023...
  -> Buscando datos del año 2022...
  -> Buscando datos del año 2021...
  -> Buscando datos del año 2020...
  -> Buscando datos del año 2019...
  -> Buscando datos del año 2018...
  -> Buscando datos del año 2017...
  -> Buscando datos del año 2016...
  -> Buscando datos del año 2015...
[+] Paso 1 completado. Usuarios semilla consolidados: 287
[+] Paso 1 guardado en: datos_euskera/01_euskera_dataset_usuarios_semilla_V3.csv


,userName,description,seed_tweet,seed_year,territorio
0,MTorezwriting,,"It's Sunday, so as usual, I ask what made you ...",2026,Euskera
1,xesusgalan,,@forls1906 @voxlacoruna A ver é que si que rab...,2026,Euskera
2,jordimurgo,,I was considering building a couple of servers...,2026,Euskera
3,_AndresBlanco_,,@ABsteward @BradSpellberg - No pathogen-specif...,2026,Euskera
4,UsuarezMD,,"IACH 🗞️ News of the Week, June 18, 2026 • @Moh...",2026,Euskera
5,LuisMix_52,,@ClarksonsFarm1 I can only wish you a speedy r...,2026,Euskera
6,aruvinchan,,I just can't imagine a life where I'm complete...,2026,Euskera
7,Manugrillo2,,These findings support #finerenone as a founda...,2026,Euskera
8,BuffaloTeo,,זהו עברתי את הגבול!!\nI'm in S pain!!!!!,2026,Euskera
9,Cinedlos80,,@CozarPablo86 Claro ho. Hasta le hice podcast....,2026,Euskera


## 2. Filtro LLM (Depuración de Cuentas Institucionales)

Para evitar la contaminación del corpus discursivo con noticias, campañas de prevención u ONGs, se integra la API de modelos de lenguaje (LLM) Google Gemini. El modelo realiza una clasificación semántica *zero-shot* evaluando la biografía y el nombre de usuario de cada perfil para garantizar que la población semilla esté compuesta predominantemente por individuos físicos reales.

**Nota metodológica:** Si un usuario carece de biografía (un comportamiento común en perfiles anónimos), el algoritmo asume por defecto la etiqueta "PERSONAL". Esta decisión estratégica prioriza el *Recall*, evitando descartar prematuramente a posibles sujetos de estudio clínico. Además, para garantizar la estabilidad de la extracción y respetar los límites de la cuota de la API gratuita (15 peticiones por minuto), el algoritmo implementa una ventana de latencia forzada (`time.sleep(4)`) entre inferencias.

In [ ]:
# =========================================================
# PASO 2. CLASIFICACIÓN DE CUENTAS
# PERSONAL vs INSTITUCIONAL
# =========================================================

import time
import pandas as pd

def classify_user_type(username, description):
    """
    Clasificación zero-shot de perfiles de X (Twitter)
    mediante Gemini 2.5 Flash.
    """
    if pd.isna(description) or not description or str(description).strip() == "Sin biografía" or str(description).strip() == "":
        return "PERSONAL"

    prompt = f"""
Actúa como un experto clasificador de perfiles de redes sociales.

A continuación tienes el nombre de usuario y la biografía de una cuenta de X (Twitter).

Tu tarea es determinar si la cuenta es:

- PERSONAL → individuo físico
- INSTITUCIONAL → empresas, ONGs, medios, asociaciones, hospitales, clínicas, partidos políticos, sindicatos, fundaciones, bots o entidades colectivas

Responde ÚNICAMENTE con:
PERSONAL
o
INSTITUCIONAL

No añadas explicaciones.

Usuario: @{username}

Biografía:
{description}
"""

    try:
        response = modelo_gemini.generate_content(prompt)
        resultado = response.text.strip().upper()

        if "INSTITUCIONAL" in resultado:
            return "INSTITUCIONAL"

        return "PERSONAL"

    except Exception:
        # En caso de error de conexión con la API, priorizamos el Recall
        return "PERSONAL"

print(
    f"[*] Paso 2: Ejecutando inferencia LLM con Gemini "
    f"para {len(df_semilla)} cuentas de {COUNTRY}..."
)

print(
    "    (Procesando a alta velocidad gracias a la cuota de API Pospago)"
)

# =========================================================
# BUCLE DE CLASIFICACIÓN (ALTA VELOCIDAD)
# =========================================================

tipos_cuenta = []

for idx, row in df_semilla.iterrows():

    # Imprimimos cada 50 porque ahora irá muy rápido
    if idx % 50 == 0 and idx > 0:
        print(f"  -> Procesados {idx}/{len(df_semilla)} usuarios...")

    tipo = classify_user_type(
        row["userName"],
        row["description"]
    )

    tipos_cuenta.append(tipo)

    # Latencia mínima para API de pago (0.5 segundos)
    time.sleep(0.5)

df_semilla["tipo_cuenta"] = tipos_cuenta

# =========================================================
# RESULTADOS
# =========================================================

print("\n[+] Paso 2 completado.")

print("\nDistribución de tipos de cuenta:")
print(df_semilla["tipo_cuenta"].value_counts())

# =========================================================
# GUARDADO DEL PASO 2 (Rutas dinámicas)
# =========================================================

ruta_paso2 = f"{OUTPUT_DIR}/02_{COUNTRY_CODE}_dataset_usuarios_semilla_clasificados.csv"

df_semilla.to_csv(
    ruta_paso2,
    index=False,
    encoding="utf-8-sig"
)

print(f"\n[✔] Paso 2 guardado en: {ruta_paso2}")

# Mostrar una muestra de cómo ha clasificado las cuentas
display(df_semilla[['userName', 'description', 'tipo_cuenta']].head(10))

[*] Paso 2: Ejecutando inferencia LLM con Gemini para 287 cuentas de Euskera...
    (Procesando a alta velocidad gracias a la cuota de API Pospago)
  -> Procesados 50/287 usuarios...
  -> Procesados 100/287 usuarios...
  -> Procesados 150/287 usuarios...
  -> Procesados 200/287 usuarios...
  -> Procesados 250/287 usuarios...

[+] Paso 2 completado.

Distribución de tipos de cuenta:
tipo_cuenta
PERSONAL    287
Name: count, dtype: int64

[✔] Paso 2 guardado en: datos_euskera/02_euskera_dataset_usuarios_semilla_clasificados.csv


,userName,description,tipo_cuenta
0,MTorezwriting,,PERSONAL
1,xesusgalan,,PERSONAL
2,jordimurgo,,PERSONAL
3,_AndresBlanco_,,PERSONAL
4,UsuarezMD,,PERSONAL
5,LuisMix_52,,PERSONAL
6,aruvinchan,,PERSONAL
7,Manugrillo2,,PERSONAL
8,BuffaloTeo,,PERSONAL
9,Cinedlos80,,PERSONAL


## 3. Agrupación y Aislamiento de Individuos Físicos

Tras la inferencia semántica ejecutada por el modelo de lenguaje, se procede a la depuración estricta del conjunto de datos. En esta fase, se aíslan exclusivamente los perfiles clasificados bajo la etiqueta "PERSONAL", descartando sistemáticamente las cuentas de carácter institucional, corporativo, informativo o automatizado (bots). El subconjunto resultante se consolida y exporta como la **Población Semilla definitiva**, sobre la cual se ejecutará la posterior extracción longitudinal de su contexto histórico y discursivo.

In [ ]:
import os
import pandas as pd

# ============================================================
# FUNCIONES AUXILIARES DE CARGA (HELPERS)
# ============================================================

def cargar_csv_si_existe(ruta):
    """Verifica si el archivo CSV existe y lo carga en un DataFrame."""
    if os.path.exists(ruta):
        return pd.read_csv(ruta)
    return None

def cargar_json_si_existe(ruta):
    """Verifica si el archivo JSON existe y lo carga en un diccionario."""
    if os.path.exists(ruta):
        import json
        with open(ruta, "r", encoding="utf-8") as f:
            return json.load(f)
    return None

# ============================================================
# 3. FILTRADO DE INDIVIDUOS FÍSICOS
# ============================================================

# Ruta de salida estandarizada utilizando la variable de país
ruta_paso3 = f"{OUTPUT_DIR}/03_{COUNTRY_CODE}_dataset_usuarios_fisicos.csv"

# Invocación de la función auxiliar recién definida
df_usuarios_finales = cargar_csv_si_existe(ruta_paso3)

if df_usuarios_finales is None:
    print(f"[*] Paso 3: Aislando individuos físicos de la muestra de {COUNTRY}...")

    # Filtrado estricto mediante máscara booleana de Pandas
    df_usuarios_finales = df_semilla[
        df_semilla["tipo_cuenta"] == "PERSONAL"
    ].copy()

    # Exportación del subconjunto limpio
    df_usuarios_finales.to_csv(
        ruta_paso3,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"[+] Paso 3 guardado exitosamente en: {ruta_paso3}")
else:
    print(f"[*] Paso 3: Cargando datos previamente filtrados desde {ruta_paso3}")

# Resumen de retención de la muestra
print(
    f"[+] Población física final consolidada: "
    f"{len(df_usuarios_finales)} perfiles retenidos de los {len(df_semilla)} iniciales."
)

# Muestra visual del DataFrame final
display(df_usuarios_finales.head())

[*] Paso 3: Aislando individuos físicos de la muestra de Euskera...
[+] Paso 3 guardado exitosamente en: datos_euskera/03_euskera_dataset_usuarios_fisicos.csv
[+] Población física final consolidada: 287 perfiles retenidos de los 287 iniciales.


,userName,description,seed_tweet,seed_year,territorio,tipo_cuenta
0,MTorezwriting,,"It's Sunday, so as usual, I ask what made you ...",2026,Euskera,PERSONAL
1,xesusgalan,,@forls1906 @voxlacoruna A ver é que si que rab...,2026,Euskera,PERSONAL
2,jordimurgo,,I was considering building a couple of servers...,2026,Euskera,PERSONAL
3,_AndresBlanco_,,@ABsteward @BradSpellberg - No pathogen-specif...,2026,Euskera,PERSONAL
4,UsuarezMD,,"IACH 🗞️ News of the Week, June 18, 2026 • @Moh...",2026,Euskera,PERSONAL


## 4. Extracción Longitudinal del Contexto Discursivo

Con la población semilla depurada (filtrada exclusivamente a individuos físicos), el *pipeline* procede a la construcción del corpus discursivo individual. Esta fase metodológica es crítica: permite transcender el análisis transversal de un mensaje aislado (*seed tweet*) y construir un contexto longitudinal. Este enfoque es fundamental en la lingüística computacional aplicada a la salud mental, ya que posibilita el modelado temático y la evaluación de riesgo psicológico sostenido en el tiempo.

El algoritmo realiza llamadas iterativas a la API para extraer un máximo de 25 publicaciones recientes por perfil. Para garantizar la viabilidad técnica de la extracción masiva y prevenir el bloqueo temporal por parte de los servidores (*Rate Limiting*), se implementa un sistema de paginación mediante cursores y un intervalo de cortesía entre peticiones. Los perfiles cuyo historial no esté disponible (cuentas suspendidas o protegidas) se capturan mediante manejo de excepciones, garantizando la continuidad del proceso.


In [ ]:
# ============================================================
# 4. EXTRACCIÓN DEL CORPUS HISTÓRICO LONGITUDINAL
# ============================================================

# Parámetro de profundidad histórica (25 tweets como contexto basal)
MAX_TWEETS_HISTORICO = 25

# Función auxiliar para cargar JSON
def cargar_json_si_existe(ruta):
    if os.path.exists(ruta):
        with open(ruta, "r", encoding="utf-8") as f:
            return json.load(f)
    return None

# Ruta dinámica para el país actual
ruta_paso4 = f"{OUTPUT_DIR}/04_{COUNTRY_CODE}_dataset_contexto_historico.json"

datos_contexto = cargar_json_si_existe(ruta_paso4)

def fetch_historical_context(users_list, api_key, max_tweets=MAX_TWEETS_HISTORICO):
    if not api_key:
        raise ValueError("Falta TWITTER_API_KEY en la configuración.")

    print(
        f"[*] Paso 4: Extrayendo corpus histórico "
        f"(hasta {max_tweets} tweets por usuario) "
        f"para {len(users_list)} individuos físicos de {COUNTRY}..."
    )

    base_url = "https://api.twitterapi.io/twitter/user/last_tweets"
    headers = {"X-API-Key": api_key}
    context_data = {}

    for idx, user in enumerate(users_list, start=1):
        if idx % 10 == 0 or idx == 1:
            print(f"  [{idx}/{len(users_list)}] Contextualizando timeline de @{user}...")

        user_tweets = []
        cursor = None

        while len(user_tweets) < max_tweets:
            params = {"userName": user}
            if cursor:
                params["cursor"] = cursor

            try:
                response = requests.get(
                    base_url,
                    headers=headers,
                    params=params,
                    timeout=60
                )

                if response.status_code != 200:
                    # Silenciamos errores menores de cuentas privadas/borradas para no ensuciar el log
                    # print(f"    [!] Error al extraer tweets de @{user} (Code: {response.status_code})")
                    break

                data = response.json()
                tweets = data.get("data", {}).get("tweets", [])
                has_next_page = data.get("has_next_page", False)
                cursor = data.get("next_cursor")

                for t in tweets:
                    texto = t.get("text")
                    if texto:
                        user_tweets.append(texto)
                    if len(user_tweets) >= max_tweets:
                        break

                if not has_next_page:
                    break

            except Exception as e:
                print(f"    [!] Excepción de red con @{user}: {e}")
                break

        context_data[user] = user_tweets

        # Latencia forzada para cuidar el límite de la API
        time.sleep(1)

    return context_data

# Ejecución del pipeline de extracción
if datos_contexto is None:
    # Extraemos la lista de la columna del dataframe generado en el Paso 3
    lista_usuarios = df_usuarios_finales["userName"].dropna().tolist()

    datos_contexto = fetch_historical_context(
        lista_usuarios,
        TWITTER_API_KEY,
        max_tweets=MAX_TWEETS_HISTORICO
    )

    # Volcado seguro a JSON para mantener la estructura de lista de tweets por usuario
    with open(ruta_paso4, "w", encoding="utf-8") as f:
        json.dump(
            datos_contexto,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(f"[+] Paso 4 guardado exitosamente en: {ruta_paso4}")
else:
    print(f"[*] Paso 4: Cargando corpus histórico previamente extraído desde {ruta_paso4}")

# Validación de completitud
print(f"[+] Usuarios con contexto histórico extraído: {len(datos_contexto)} de {len(df_usuarios_finales)} intentados.")

## 5. Anonimización y Cumplimiento Ético (Data Sanitization)

Para cumplir rigurosamente con los estándares éticos de investigación universitaria y la normativa de protección de datos (RGPD), se aplica un *pipeline* de Procesamiento de Lenguaje Natural impulsado por Modelos de Reconocimiento de Entidades Nombradas (spaCy NER), combinado con reglas heurísticas de Expresiones Regulares (Regex).

Este proceso enmascara de forma automatizada cualquier Información de Identificación Personal (PII) presente en los textos —incluyendo nombres propios, ubicaciones, menciones a otros usuarios y números telefónicos o identificadores—. Adicionalmente, el identificador único original del usuario es sustituido por un *hash* criptográfico unidireccional (SHA-256), garantizando la privacidad absoluta de los sujetos de estudio antes del almacenamiento analítico de los datos.

In [ ]:
import hashlib # Importación crucial para la criptografía de los usuarios

# ============================================================
# 5. ANONIMIZACIÓN ÉTICA DEL CORPUS
# ============================================================

# Ruta dinámica para el país actual
ruta_paso5 = f"{OUTPUT_DIR}/05_{COUNTRY_CODE}_dataset_anonimizado.json"

dataset_anonimizado = cargar_json_si_existe(ruta_paso5)

if dataset_anonimizado is None:
    print(f"[*] Paso 5: Desplegando pipeline de anonimización NLP para {COUNTRY}...")

    # Carga del modelo NLP en español
    nlp = spacy.load("es_core_news_md")
    scrubber = scrubadub.Scrubber()

    # Función 1: Enmascaramiento criptográfico del nombre de usuario
    def generar_usuario_anonimo(user):
        digest = hashlib.sha256(str(user).encode("utf-8")).hexdigest()
        return f"USER_{digest[:10]}"

    # Función 2: Limpieza de PII en el texto del tweet
    def anonymize_text(text):
        if not isinstance(text, str):
            return ""

        # Limpieza mediante Expresiones Regulares
        text = re.sub(r"https?://\S+|www\.\S+", "{{URL}}", text)
        text = re.sub(r"@\w+", "{{USER_MENTION}}", text)
        text = re.sub(r"\d{4,}", "{{NUMBER}}", text)

        # Limpieza mediante spaCy NER (Reconocimiento de Entidades Nombradas)
        doc = nlp(text)
        new_text_parts = []
        last_idx = 0

        for ent in doc.ents:
            if ent.label_ in ("LOC", "GPE"): # Localizaciones
                new_text_parts.append(text[last_idx:ent.start_char])
                new_text_parts.append("{{LOCATION}}")
                last_idx = ent.end_char

            elif ent.label_ in ("PER", "PERSON"): # Nombres de personas
                new_text_parts.append(text[last_idx:ent.start_char])
                new_text_parts.append("{{PERSON}}")
                last_idx = ent.end_char

        text = "".join(new_text_parts) + text[last_idx:]

        # Capa de seguridad adicional con Scrubadub
        text = scrubber.clean(text)

        return text

    # Ejecución del pipeline sobre el contexto histórico
    dataset_anonimizado = {}

    for user, tweets in datos_contexto.items():
        user_anon = generar_usuario_anonimo(user)

        tweets_limpios = [
            anonymize_text(t)
            for t in tweets
        ]

        dataset_anonimizado[user_anon] = tweets_limpios

    # Volcado de datos seguros
    with open(ruta_paso5, "w", encoding="utf-8") as f:
        json.dump(
            dataset_anonimizado,
            f,
            indent=4,
            ensure_ascii=False
        )

    print(f"[+] Paso 5 guardado exitosamente en: {ruta_paso5}")
else:
    print(f"[*] Paso 5: Cargando dataset previamente anonimizado desde {ruta_paso5}")

print(f"[+] Usuarios enmascarados y asegurados: {len(dataset_anonimizado)}")

# Muestra visual de la auditoría de anonimización
print("\n[Auditoría de Enmascaramiento - Muestra Aleatoria]")
for usuario, tweets in list(dataset_anonimizado.items())[:3]:
    print(f"\n=== Historial de {usuario} ===")
    for i, tweet in enumerate(tweets[:3], start=1):
        print(f"  [{i}] {tweet}")

[*] Paso 5: Desplegando pipeline de anonimización NLP para Euskera...
[+] Paso 5 guardado exitosamente en: datos_euskera/05_euskera_dataset_anonimizado.json
[+] Usuarios enmascarados y asegurados: 287

[Auditoría de Enmascaramiento - Muestra Aleatoria]

=== Historial de USER_698f0f2445 ===
  [1] RT {{USER_MENTION}}: good things look good on grateful people.
  [2] Great track from a classic album! #rnb #ilovemusic {{USER_MENTION}}

{{URL}}
  [3] It's Sunday, so as usual, I ask what made you happy this week. Getting into a habit of being grateful and looking for positives saves lives. {{PERSON}} was locked up and being abused I had to cling to the seemingly "small" things in life to keep me alive; a warm shower, a nice coffee, the smell of a new magazine.
Comment below what made you happy 

#mentalhealth #motivation #sundayvibes

=== Historial de USER_8a9dda3c02 ===
  [1] RT {{USER_MENTION}}: Por aquel entonces no llegaban tantas lecciones de moral sobre los filiales... Aún sabiendo que 

## 6. Filtro Semántico (Salud Mental vs. Ruido)



En este paso final se utiliza un Modelo de Lenguaje de Gran Escala (LLM) para identificar qué publicaciones del dataset están genuinamente relacionadas con la salud mental (depresión, malestar emocional, riesgo suicida) y cuáles constituyen ruido coloquial.

**Decisiones Metodológicas:** Aunque en fases iniciales del pipeline se aplicó un filtrado por palabras clave, el lenguaje natural es intrínsecamente ambiguo. El LLM actúa como un filtro semántico avanzado capaz de discriminar usos literales de expresiones figuradas (por ejemplo, diferenciar entre "me muero de risa" o "qué depresión de partido"). El objetivo prioritario de esta fase no es el diagnóstico clínico, sino la depuración del ruido analítico, etiquetando cada tweet de forma binaria: 1 (Riesgo/Salud Mental) o 0 (Ruido/Coloquial).



### 6.1 Carga de Datos y Definición del Clasificador



Se realiza la carga del corpus anonimizado generado en la fase anterior y se define la función de clasificación *Zero-Shot* parametrizada que procesará las secuencias discursivas de los timelines.

In [ ]:
# =========================================================
# PASO 6.1. CARGA Y FUNCIÓN DE CLASIFICACIÓN SEMÁNTICA
# =========================================================

import os
import json
import time
import pandas as pd
import google.generativeai as genai
from google.generativeai.types import HarmCategory, HarmBlockThreshold

# =========================================================
# CARGA DEL DATASET ANONIMIZADO (Ruta dinámica)
# =========================================================

ruta_archivo = f"{OUTPUT_DIR}/05_{COUNTRY_CODE}_dataset_anonimizado.json"

with open(ruta_archivo, "r", encoding="utf-8") as f:
    dataset_anonimizado = json.load(f)

print(
    f"[*] Paso 6.1: Datos cargados para {COUNTRY}. "
    f"Usuarios a analizar: {len(dataset_anonimizado)}"
)

# =========================================================
# CONFIGURACIÓN DE GEMINI
# =========================================================

# Anulamos proxies internos de Colab que causan bloqueos
os.environ['HTTP_PROXY'] = ''
os.environ['HTTPS_PROXY'] = ''

genai.configure(api_key=GEMINI_API_KEY, transport="rest")
modelo_gemini = genai.GenerativeModel("gemini-2.5-flash")

# Apagamos los filtros de seguridad para evitar bloqueos por contenido sensible
safety_settings = {
    HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
    HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
    HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
    HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
}

# =========================================================
# FUNCIÓN DE CLASIFICACIÓN SEMÁNTICA
# =========================================================

def clasificar_tweets_salud_mental(lista_tweets, max_reintentos=3):
    """
    Clasificador semántico con sistema antibloqueo y manejo de cuotas.

    1 = Contenido relacionado con depresión, suicidio, autolesión o desesperanza.
    0 = Contenido no relacionado, coloquial, humorístico, figurado, informativo o ambiguo.
    """
    if not lista_tweets:
        return []

    tweets_formateados = "\n".join(
        [f"{i + 1}. {tweet}" for i, tweet in enumerate(lista_tweets)]
    )

    prompt = f"""
Actúa como experta en análisis semántico de discurso en redes sociales.

Vas a recibir una lista de tweets anonimizados de un mismo usuario.
Tu tarea es clasificar CADA tweet individualmente.

Criterio:

Etiqueta con 1 si el tweet expresa o menciona de forma clara alguno de estos elementos:
- depresión
- desesperanza
- deseo de morir
- ideación suicida
- intento suicida
- autolesión
- deseo de desaparecer
- sensación de vacío extremo
- sufrimiento emocional intenso asociado a no querer seguir viviendo

Etiqueta con 0 si:
- no está relacionado con depresión, suicidio o autolesión
- menciona salud mental de forma genérica, institucional, divulgativa o informativa
- el uso es coloquial, humorístico, metafórico o ambiguo
- habla de ansiedad, estrés o malestar general sin relación clara con depresión o suicidio
- no hay suficiente evidencia textual

Importante:
- No hagas diagnóstico clínico.
- Evalúa solo el contenido textual del tweet.
- No infieras información que no esté explícita o semánticamente clara.
- Responde SOLO en formato JSON válido.
- Devuelve exactamente una etiqueta por tweet.

Formato requerido:
[
  {{"tweet_id": 1, "label": 0}},
  {{"tweet_id": 2, "label": 1}}
]

Tweets:
{tweets_formateados}
"""

    for intento in range(max_reintentos):
        try:
            # Forzamos un timeout de 20s para que no se quede colgado eternamente
            response = modelo_gemini.generate_content(
                prompt,
                safety_settings=safety_settings,
                request_options={"timeout": 20}
            )

            texto = (
                response.text
                .replace("```json", "")
                .replace("```", "")
                .strip()
            )

            resultado = json.loads(texto)

            etiquetas = [
                item["label"]
                for item in resultado
            ]

            if len(etiquetas) != len(lista_tweets):
                print(
                    "    [!] Número de etiquetas inconsistente. "
                    "Se devolverán 0 por seguridad."
                )
                return [0] * len(lista_tweets)

            return [
                1 if x == 1 else 0
                for x in etiquetas
            ]

        except Exception as e:
            error_str = str(e).lower()

            if "429" in error_str or "quota" in error_str or "timeout" in error_str:
                print(
                    f"    [!] Límite de cuota o Timeout alcanzado. "
                    f"Pausando 5 segundos "
                    f"(Intento {intento + 1}/{max_reintentos})..."
                )
                time.sleep(5)
            else:
                print(f"    [!] Error en clasificación LLM: {e}")
                return [0] * len(lista_tweets)

    return [0] * len(lista_tweets)

[*] Paso 6.1: Datos cargados para Euskera. Usuarios a analizar: 287


### 6.2 Inferencia Semántica Masiva



Se ejecuta la iteración controlada sobre el corpus anonimizado de perfiles personales. El algoritmo envía el bloque completo de publicaciones por individuo para maximizar la eficiencia del contexto del modelo y respetar las restricciones de tráfico de la ventana temporal.

In [ ]:
# =========================================================
# PASO 6.2. CLASIFICACIÓN SEMÁNTICA DEL DATASET
# =========================================================

print(f"[*] Paso 6.2: Iniciando clasificación semántica del dataset de {COUNTRY}...")

dataset_clasificado = {}
total_usuarios = len(dataset_anonimizado)

for idx, (usuario, tweets) in enumerate(
    dataset_anonimizado.items(),
    start=1
):
    if idx % 10 == 0 or idx == 1:
        print(f"  -> Procesados {idx}/{total_usuarios} usuarios...")

    etiquetas = clasificar_tweets_salud_mental(tweets)

    dataset_clasificado[usuario] = [
        {
            "tweet": tweet,
            "mental_health": etiqueta
        }
        for tweet, etiqueta in zip(tweets, etiquetas)
    ]

    # Ventana de latencia de seguridad obligatoria para el Rate Limiting de Gemini
    time.sleep(6)

print("\n[+] Clasificación completada exitosamente.")

### 6.3 Consolidación, Resumen Cuantitativo y Auditoría



Tras la asignación de etiquetas semánticas, se estructuran los conjuntos de datos de salida definitivos en formato estructurado (JSON completo para modelados textuales ulteriores y CSV resumido para análisis estadísticos de prevalencia). El paso concluye con un control manual de calidad automatizado para validar los aciertos del modelo LLM.

In [ ]:
# =========================================================
# PASO 6.3. CONSOLIDACIÓN, MÉTRICAS Y AUDITORÍA
# =========================================================

print(f"[*] Paso 6.3: Consolidando resultados y generando métricas para {COUNTRY}...")

# =========================================================
# 1. GUARDADO DEL DATASET COMPLETO (JSON estructurado)
# =========================================================
ruta_json = f"{OUTPUT_DIR}/06_{COUNTRY_CODE}_dataset_clasificado_salud_mental.json"

with open(ruta_json, "w", encoding="utf-8") as f:
    json.dump(
        dataset_clasificado,
        f,
        ensure_ascii=False,
        indent=2
    )

# =========================================================
# 2. GENERACIÓN DEL RESUMEN CUANTITATIVO (Métricas de prevalencia)
# =========================================================
resumen = []

for usuario, tweets in dataset_clasificado.items():
    total = len(tweets)
    mh = sum(t["mental_health"] for t in tweets)

    resumen.append({
        "usuario": usuario,
        "total_tweets": total,
        "tweets_salud_mental": mh,
        "tweets_no_salud_mental": total - mh,
        "proporcion_salud_mental": round(mh / total if total > 0 else 0, 3)
    })

df_resumen = pd.DataFrame(resumen)
ruta_csv = f"{OUTPUT_DIR}/06_{COUNTRY_CODE}_resumen_clasificacion_salud_mental.csv"

df_resumen.to_csv(
    ruta_csv,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"[+] Archivos exportados de forma segura:\n"
    f"  - JSON Completo: {ruta_json}\n"
    f"  - CSV Estadístico: {ruta_csv}"
)

print(
    f"[*] Media global de proporción de tweets de riesgo para {COUNTRY}: "
    f"{df_resumen['proporcion_salud_mental'].mean():.3f}\n"
)

# =========================================================
# 3. AUDITORÍA DE SEGURIDAD (Control visual de verdaderos positivos)
# =========================================================
print("=" * 60)
print("AUDITORÍA DE RIESGO: Muestra de tweets etiquetados como '1'")
print("=" * 60)

hits = 0

for usuario, tweets in dataset_clasificado.items():
    for item in tweets:
        if item["mental_health"] == 1:
            print(f"[{usuario}] → {item['tweet']}\n")
            hits += 1

            if hits >= 20:
                break
    if hits >= 20:
        break

if hits == 0:
    print(
        "⚠️ Control: No se ha detectado ningún tweet clasificado "
        "con riesgo (Label=1)."
    )

# =========================================================
# 4. VISTA RÁPIDA DEL DATAFRAME RESUMEN
# =========================================================
display(df_resumen.head())

In [ ]:
# =========================================================
# PASO 6.3. CONSOLIDACIÓN, MÉTRICAS Y AUDITORÍA
# =========================================================

print(f"[*] Paso 6.3: Consolidando resultados y generando métricas para {COUNTRY}...")

# =========================================================
# 1. GUARDADO DEL DATASET COMPLETO (JSON estructurado)
# =========================================================
ruta_json = f"{OUTPUT_DIR}/06_{COUNTRY_CODE}_dataset_clasificado_salud_mental.json"

with open(ruta_json, "w", encoding="utf-8") as f:
    json.dump(
        dataset_clasificado,
        f,
        ensure_ascii=False,
        indent=2
    )

# =========================================================
# 2. GENERACIÓN DEL RESUMEN CUANTITATIVO (Métricas de prevalencia)
# =========================================================
resumen = []

for usuario, tweets in dataset_clasificado.items():
    total = len(tweets)
    mh = sum(t["mental_health"] for t in tweets)

    resumen.append({
        "usuario": usuario,
        "total_tweets": total,
        "tweets_salud_mental": mh,
        "tweets_no_salud_mental": total - mh,
        "proporcion_salud_mental": round(mh / total if total > 0 else 0, 3)
    })

df_resumen = pd.DataFrame(resumen)
ruta_csv = f"{OUTPUT_DIR}/06_{COUNTRY_CODE}_resumen_clasificacion_salud_mental.csv"

df_resumen.to_csv(
    ruta_csv,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"[+] Archivos exportados de forma segura:\n"
    f"  - JSON Completo: {ruta_json}\n"
    f"  - CSV Estadístico: {ruta_csv}"
)

print(
    f"[*] Media global de proporción de tweets de riesgo para {COUNTRY}: "
    f"{df_resumen['proporcion_salud_mental'].mean():.3f}\n"
)

# =========================================================
# 3. AUDITORÍA DE SEGURIDAD (Control visual de verdaderos positivos)
# =========================================================
print("=" * 60)
print("AUDITORÍA DE RIESGO: Muestra de tweets etiquetados como '1'")
print("=" * 60)

hits = 0

for usuario, tweets in dataset_clasificado.items():
    for item in tweets:
        if item["mental_health"] == 1:
            print(f"[{usuario}] → {item['tweet']}\n")
            hits += 1

            if hits >= 20:
                break
    if hits >= 20:
        break

if hits == 0:
    print(
        "⚠️ Control: No se ha detectado ningún tweet clasificado "
        "con riesgo (Label=1)."
    )

# =========================================================
# 4. VISTA RÁPIDA DEL DATAFRAME RESUMEN
# =========================================================
display(df_resumen.head())

## 7. Agregación y Clasificación de Usuarios (Riesgo vs. Control)


En este paso, los resultados de la inferencia semántica binaria (Paso 6) se escalan a nivel de usuario. Para evitar la sobreinterpretación analítica de publicaciones aisladas o descontextualizadas (sesgo de falso positivo por un único *tweet*), se establece una regla de decisión heurística basada en la densidad del discurso.

**Decisiones Metodológicas:**
Un usuario es clasificado en el grupo de riesgo empírico ("seleccionado") exclusivamente si cumple dos criterios de forma simultánea:
1. **Volumen mínimo absoluto:** Una cantidad base de publicaciones validadas por el LLM como relacionadas con sufrimiento emocional o ideación suicida.
2. **Proporción umbral relativa:** Un porcentaje mínimo de este tipo de contenido respecto a su corpus longitudinal total extraído.

Los perfiles que no superan este umbral analítico son asignados al grupo "control". Adicionalmente, el algoritmo aísla el **"corpus cotidiano"** de los individuos de riesgo (es decir, extrayendo únicamente sus *tweets* etiquetados con 0). Esto resulta fundamental para permitir posteriores análisis lingüísticos sobre el discurso aparentemente "neutro" o funcional de las personas que sufren ideación suicida.

In [ ]:
# ============================================================
# 7. SELECCIÓN DE USUARIOS Y CORPUS COTIDIANO
# ============================================================

# UMBRALES METODOLÓGICOS DE RIESGO (Ajustables según necesidad del investigador)
MIN_TWEETS_MH = 2        # Mínimo de tweets detectados con riesgo
MIN_PROPORCION_MH = 0.10 # Mínimo 10% del corpus del usuario debe ser de riesgo

# Rutas dinámicas para la exportación de artefactos
ruta_paso7 = f"{OUTPUT_DIR}/07_{COUNTRY_CODE}_usuarios_clasificados_seleccionados_control.csv"
ruta_cotidiano = f"{OUTPUT_DIR}/07_{COUNTRY_CODE}_corpus_cotidiano_usuarios_seleccionados.csv"

# Funciones de carga por si la celda se ejecuta repetidas veces
df_usuarios = cargar_csv_si_existe(ruta_paso7)
df_cotidiano = cargar_csv_si_existe(ruta_cotidiano)

if df_usuarios is None:
    print(f"[*] Paso 7: Consolidando clasificación a nivel de usuario para {COUNTRY}...")

    datos_usuarios = []

    for usuario, tweets in dataset_clasificado.items():
        total_tweets = len(tweets)

        if total_tweets == 0:
            continue

        total_mh = sum(t["mental_health"] for t in tweets)
        proporcion_mh = total_mh / total_tweets

        # Aplicación estricta de la heurística de riesgo
        etiqueta = (
            "seleccionado"
            if (total_mh >= MIN_TWEETS_MH and proporcion_mh >= MIN_PROPORCION_MH)
            else "control"
        )

        datos_usuarios.append({
            "territorio": COUNTRY,
            "usuario": usuario,
            "total_tweets": total_tweets,
            "tweets_salud_mental": total_mh,
            "proporcion_salud_mental": round(proporcion_mh, 3),
            "clasificacion_usuario": etiqueta
        })

    df_usuarios = pd.DataFrame(datos_usuarios)

    df_usuarios.to_csv(
        ruta_paso7,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"[+] Paso 7 (Métricas de Usuario) guardado en: {ruta_paso7}")
else:
    print(f"[*] Paso 7: Cargando métricas de usuarios previamente clasificadas desde {ruta_paso7}")

print("\n[+] Distribución final de usuarios (Riesgo vs Control):")
print(df_usuarios["clasificacion_usuario"].value_counts().to_string())

# ============================================================
# EXTRACCIÓN DEL CORPUS COTIDIANO (Discurso neutro de la muestra de riesgo)
# ============================================================

if df_cotidiano is None:
    print(f"\n[*] Preparando corpus cotidiano de los usuarios seleccionados en {COUNTRY}...")

    # Identificamos qué usuarios entraron al grupo de riesgo empírico
    usuarios_seleccionados = set(
        df_usuarios.loc[
            df_usuarios["clasificacion_usuario"] == "seleccionado",
            "usuario"
        ]
    )

    filas_cotidianas = []

    for usuario, tweets in dataset_clasificado.items():
        # Descartamos a los del grupo control para este corpus específico
        if usuario not in usuarios_seleccionados:
            continue

        for item in tweets:
            # Extraemos EXCLUSIVAMENTE los tweets etiquetados como 0 (neutros/coloquiales)
            if int(item.get("mental_health", 0)) == 0:
                filas_cotidianas.append({
                    "territorio": COUNTRY,
                    "usuario": usuario,
                    "tweet": item.get("tweet", "")
                })

    df_cotidiano = pd.DataFrame(filas_cotidianas)

    df_cotidiano.to_csv(
        ruta_cotidiano,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"[+] Corpus cotidiano exportado en: {ruta_cotidiano}")
else:
    print(f"[*] Cargando corpus cotidiano previamente extraído desde {ruta_cotidiano}")

print(f"\n[+] Total de tweets cotidianos disponibles para análisis (Paso 8): {len(df_cotidiano)}")
display(df_cotidiano.head())

## 8. Identificación de Microtemas en el Corpus Cotidiano




En esta fase analítica, se procesa exclusivamente el "corpus cotidiano" (los *tweets* etiquetados como 0 o "neutros") de aquellos usuarios previamente identificados en el grupo de riesgo empírico. El objetivo es realizar un perfilado temático (*topic modeling* supervisado) para descubrir sobre qué hablan estos individuos cuando no expresan malestar psicológico.

### 8.1 Preparación del Corpus para la Clasificación



Antes de la inferencia, se aísla y prepara estructuralmente el corpus de entrada. Se asigna un identificador local único a cada *tweet* para garantizar la trazabilidad durante la respuesta estructurada en JSON que devolverá el Modelo de Lenguaje.

In [ ]:
# ============================================================
# 8.1 PREPARACIÓN DEL CORPUS PARA LA CLASIFICACIÓN
# ============================================================

import pandas as pd

print(f"[*] Paso 8.1: Preparación del corpus microtemático para {COUNTRY}...")

# Validación de seguridad del entorno
if "df_cotidiano" not in globals() or df_cotidiano.empty:
    raise ValueError("[!] No se encontró df_cotidiano. Ejecuta el Paso 7 primero.")

# Creación del dataframe base de trabajo
df_base_micro = df_cotidiano.copy().reset_index(drop=True)

# Asignación de ID local para el parseo del LLM
df_base_micro["tweet_id_local"] = df_base_micro.index + 1

# Auditoría de la estructura preparada
print(f"[+] Tweets cotidianos listos para analizar: {len(df_base_micro)}")
print(f"[+] Usuarios únicos en la muestra: {df_base_micro['usuario'].nunique()}")
print(f"[+] Columnas disponibles: {list(df_base_micro.columns)}")

display(df_base_micro.head())

### 8.2 Inferencia Microtemática Masiva



Con el corpus preparado, la clasificación se realiza agrupando los datos a nivel de usuario. El LLM evalúa el contexto de la lista de publicaciones y asigna una única etiqueta ("Microtema") a cada una, optimizando así el consumo de *tokens* y el tiempo de ejecución.

In [ ]:
# ============================================================
# 8.2 INFERENCIA MICROTEMÁTICA DEL CORPUS
# ============================================================

import json
import time

print(f"[*] Paso 8.2: Iniciando extracción de microtemas mediante LLM (Alta Velocidad)...")

def clasificar_microtemas(lista_tweets, max_reintentos=3):
    if not lista_tweets:
        return []

    tweets_formateados = "\n".join([f"{i + 1}. {tweet}" for i, tweet in enumerate(lista_tweets)])

    prompt = f"""
Actúa como sociólogo computacional experto en análisis temático.
A continuación tienes una lista de tweets "cotidianos" de un usuario.
Asigna un único MICROTEMA breve y concreto a cada tweet.

Reglas:
1. Devuelve un microtema conciso (ej: "Deportes", "Queja laboral", "Estudios", "Humor", "Política").
2. Si solo hay un link, emojis sin contexto o está vacío, usa estrictamente la etiqueta: "Contenido no interpretable".
3. No menciones salud mental ni depresión (eso ya se filtró).
4. Devuelve SOLO formato JSON válido.

Ejemplo de respuesta:
[
  {{"tweet_id": 1, "microtema": "Cine y series"}},
  {{"tweet_id": 2, "microtema": "Contenido no interpretable"}}
]

Tweets a analizar:
{tweets_formateados}
"""

    for intento in range(max_reintentos):
        try:
            response = modelo_gemini.generate_content(prompt)
            texto = response.text.replace("```json", "").replace("```", "").strip()
            resultado = json.loads(texto)

            temas = [item.get("microtema", "Contenido no interpretable") for item in resultado]

            if len(temas) != len(lista_tweets):
                return ["Error de parseo"] * len(lista_tweets)

            return temas

        except Exception as e:
            if "429" in str(e) or "Quota" in str(e):
                print(f"    [!] Cuota excedida. Pausa 10s (Intento {intento+1}/{max_reintentos})")
                time.sleep(10) # Pausa mucho menor, ya que los límites de pago se refrescan más rápido
            else:
                return ["Error de parseo"] * len(lista_tweets)

    return ["Error de parseo"] * len(lista_tweets)

# ============================================================
# EJECUCIÓN DEL BUCLE POR USUARIO
# ============================================================

resultados_microtemas = []
grupos = df_base_micro.groupby("usuario", sort=False)

for idx, (usuario, grupo) in enumerate(grupos, start=1):
    if idx % 10 == 0 or idx == 1: # Actualizamos cada 10 usuarios porque irá más rápido
        print(f"  -> Procesando usuario {idx}/{len(grupos)}...")

    tweets_lista = grupo["tweet"].tolist()
    temas = clasificar_microtemas(tweets_lista)

    for tweet, tema in zip(tweets_lista, temas):
        resultados_microtemas.append({
            "territorio": COUNTRY,
            "usuario": usuario,
            "tweet": tweet,
            "microtema": str(tema).strip().capitalize()
        })

    # Protección mínima de Rate Limit para capa Pospago (Nivel 1)
    time.sleep(1)

df_tweets_microtemas = pd.DataFrame(resultados_microtemas)
print("\n[+] Inferencia microtemática completada a alta velocidad.")

### 8.3 Consolidación, Corrección y Exportación



Se ejecuta una limpieza final para unificar los posibles errores de *parseo* de la API bajo la categoría "Contenido no interpretable". Posteriormente, se calculan las métricas de frecuencia temáticas (absolutas y relativas) y se exportan los artefactos en formato `CSV`.

In [ ]:
# ============================================================
# 8.3 CONSOLIDACIÓN Y EXPORTACIÓN DE RESULTADOS
# ============================================================

print(f"[*] Paso 8.3: Generando métricas y guardando resultados de {COUNTRY}...")

# 1. Corrección de errores residuales de la API
df_tweets_microtemas.loc[
    df_tweets_microtemas["microtema"].isin(["Error de parseo", "Error de parseo.", ""]),
    "microtema"
] = "Contenido no interpretable"

# 2. Cálculo de frecuencias relativas y absolutas
df_frecuencia = df_tweets_microtemas["microtema"].value_counts().reset_index()
df_frecuencia.columns = ["microtema", "frecuencia"]
df_frecuencia["porcentaje"] = (df_frecuencia["frecuencia"] / len(df_tweets_microtemas) * 100).round(2)

# 3. Rutas dinámicas de guardado
ruta_microtemas = f"{OUTPUT_DIR}/08_{COUNTRY_CODE}_tweets_cotidianos_microtemas.csv"
ruta_frecuencias = f"{OUTPUT_DIR}/08_{COUNTRY_CODE}_frecuencia_microtemas.csv"

# 4. Exportación
df_tweets_microtemas.to_csv(ruta_microtemas, index=False, encoding="utf-8-sig")
df_frecuencia.to_csv(ruta_frecuencias, index=False, encoding="utf-8-sig")

print(f"[+] Archivos exportados de forma segura:\n  - Detalle: {ruta_microtemas}\n  - Frecuencias: {ruta_frecuencias}\n")

# 5. Auditoría visual de los temas más recurrentes
print(f"[*] Top 15 Microtemas detectados en {COUNTRY}:")
display(df_frecuencia.head(15))

## 9. Normalización de Microtemas en Macrotemas


En esta fase casi final del pipeline de procesamiento de lenguaje natural, se procede a la agregación taxonómica. Los múltiples "microtemas" generados orgánicamente en el paso anterior presentan una alta dispersión semántica. Para viabilizar el análisis cuantitativo y permitir la comparabilidad sociológica entre distintos territorios, se agrupan bajo una **Taxonomía Controlada de Macrotemas**.



### 9.1 Definición de la Taxonomía y Mapeo Semántico (LLM)



Se define un esquema cerrado de categorías analíticas (ej. Salud, Ocio, Política, Relaciones). El Modelo de Lenguaje evalúa el inventario de microtemas únicos detectados en el corpus y asigna a cada uno el código macrotemático correspondiente.

In [ ]:
# ============================================================
# 9.1 TAXONOMÍA CONTROLADA Y MAPEO SEMÁNTICO MEDIANTE LLM
# ============================================================

import json
import time
import pandas as pd

print(f"[*] Paso 9.1: Iniciando normalización macrotemática para {COUNTRY}...")

# Validación del dataset previo
if "df_tweets_microtemas" not in globals() or df_tweets_microtemas.empty:
    raise ValueError("[!] No se encontró df_tweets_microtemas. Ejecuta el Paso 8 primero.")

# Taxonomía fija (Asegura la comparabilidad entre diferentes países)
TAXONOMIA_MACROTEMAS = {
    "REL": "Relaciones personales, afectivas y vida social",
    "VIDA": "Rutinas, experiencias cotidianas y estados personales",
    "OCIO": "Cultura, ocio y entretenimiento",
    "DEP": "Deporte y competición",
    "POL": "Política, instituciones y actualidad pública",
    "SERV": "Servicios públicos, administración y derechos sociales",
    "ECO": "Economía, empleo y vivienda",
    "EDU": "Educación y formación",
    "SALUD_COT": "Salud, cuerpo y bienestar cotidiano",
    "TERR": "Territorio, identidad local y comunidad",
    "MOV": "Movilidad, transporte y espacio urbano",
    "MEDIOS": "Medios, redes sociales y comunicación digital",
    "CONSUMO": "Consumo, tecnología y compras",
    "HUMOR": "Humor, memes y contenido lúdico",
    "NOINT": "Contenido no interpretable",
    "OTROS": "Otros temas o contenido ambiguo"
}

# 1. Extraemos los microtemas únicos para no enviar duplicados a la API
microtemas_unicos = df_tweets_microtemas["microtema"].unique().tolist()

# Separamos los no interpretables para no gastar tokens analizándolos
if "Contenido no interpretable" in microtemas_unicos:
    microtemas_unicos.remove("Contenido no interpretable")

print(f"[+] Microtemas únicos a clasificar: {len(microtemas_unicos)}")

def mapear_macrotemas(lista_microtemas, max_reintentos=3):
    taxonomia_texto = "\n".join([f"- {k}: {v}" for k, v in TAXONOMIA_MACROTEMAS.items()])
    microtemas_texto = "\n".join([f"- {m}" for m in lista_microtemas])

    prompt = f"""
Actúa como clasificador taxonómico. Asigna cada microtema a un código de macrotema.

Taxonomía estricta permitida:
{taxonomia_texto}

Reglas:
1. Analiza cada microtema y asígnale el CÓDIGO (ej: "REL", "OCIO") que mejor encaje.
2. Si no encaja en ninguno, usa "OTROS".
3. Devuelve estrictamente un diccionario JSON donde la clave es el microtema y el valor es el código del macrotema.

Ejemplo de salida:
{{
  "Queja laboral": "ECO",
  "Partido de fútbol": "DEP",
  "Cena con amigos": "REL"
}}

Microtemas a clasificar:
{microtemas_texto}
"""

    for intento in range(max_reintentos):
        try:
            # Enviamos TODO el inventario de golpe (aprovechando la API de pago)
            response = modelo_gemini.generate_content(prompt)
            texto = response.text.replace("```json", "").replace("```", "").strip()
            return json.loads(texto)
        except Exception as e:
            if "429" in str(e) or "Quota" in str(e):
                print(f"    [!] Cuota excedida. Pausa 10s (Intento {intento+1}/{max_reintentos})...")
                time.sleep(10)
            else:
                print(f"    [!] Error de parseo: {e}")

    return {}

# Ejecutamos el mapeo LLM
diccionario_mapeo = mapear_macrotemas(microtemas_unicos)

# Forzamos la regla de contenido no interpretable
diccionario_mapeo["Contenido no interpretable"] = "NOINT"
print("[+] Mapeo LLM completado.")

### 9.2 Consolidación del Corpus Temático y Resumen Cuantitativo



Una vez obtenido el diccionario de mapeo, se transfiere la categoría macrotemática a cada una de las publicaciones individuales del corpus. Finalmente, se agregan los datos para exportar un archivo descriptivo (`CSV`) que detalla el volumen y la distribución porcentual de cada temática a nivel nacional.

In [ ]:
# ============================================================
# 9.2 CONSOLIDACIÓN, AGREGACIÓN Y EXPORTACIÓN
# ============================================================

print(f"[*] Paso 9.2: Consolidando corpus temático final para {COUNTRY}...")

df_corpus_final = df_tweets_microtemas.copy()

# 1. Mapeo directo y ultrarrápido usando Pandas
df_corpus_final["macrotema_codigo"] = df_corpus_final["microtema"].map(diccionario_mapeo).fillna("OTROS")

# 2. Añadimos el nombre completo del macrotema
df_corpus_final["macrotema"] = df_corpus_final["macrotema_codigo"].map(TAXONOMIA_MACROTEMAS)

# 3. Exportación del Corpus a nivel Tweet
ruta_corpus = f"{OUTPUT_DIR}/09_{COUNTRY_CODE}_corpus_tematico_final.csv"
df_corpus_final.to_csv(ruta_corpus, index=False, encoding="utf-8-sig")

# 4. Creación del Resumen Agregado (Distribución de Macrotemas)
df_resumen = df_corpus_final.groupby(["macrotema_codigo", "macrotema"]).agg(
    n_tweets=("tweet", "size"),
    n_usuarios_unicos=("usuario", "nunique")
).reset_index()

df_resumen["porcentaje_tweets"] = (df_resumen["n_tweets"] / len(df_corpus_final) * 100).round(2)
df_resumen = df_resumen.sort_values("n_tweets", ascending=False).reset_index(drop=True)

# 5. Exportación del Resumen Estadístico
ruta_resumen = f"{OUTPUT_DIR}/09_{COUNTRY_CODE}_resumen_distribucion_macrotemas.csv"
df_resumen.to_csv(ruta_resumen, index=False, encoding="utf-8-sig")

print(f"[+] Archivos finales exportados de forma segura:\n  - Corpus: {ruta_corpus}\n  - Resumen: {ruta_resumen}\n")

# Auditoría Visual
print(f"[*] Distribución de Macrotemas en {COUNTRY}:")
display(df_resumen)

## 10. Análisis Cualitativo de los Principales Ejes Temáticos



El presente apartado trasciende la estadística descriptiva para abordar cómo se construyen discursivamente los macrotemas principales. El objetivo es proporcionar profundidad hermenéutica al estudio, superando la limitación del análisis puramente cuantitativo.



### 10.1 Selección de Macrotemas Relevantes y Muestreo



Para garantizar la viabilidad y pertinencia del análisis cualitativo, se filtran las categorías residuales o no interpretables. Posteriormente, se aíslan los macrotemas con mayor prevalencia en el corpus y se extrae una muestra aleatoria de publicaciones representativas, evitando el sesgo de selección manual.

In [ ]:
# ============================================================
# 10.1 SELECCIÓN DE MACROTEMAS Y PREPARACIÓN DE MUESTRAS
# ============================================================

import pandas as pd

print(f"[*] Paso 10.1: Preparando datos para el análisis cualitativo en {COUNTRY}...")

# Validación de seguridad
if "df_corpus_final" not in globals() or df_corpus_final.empty:
    raise ValueError("[!] No se encontró df_corpus_final. Ejecuta el Paso 9 primero.")

# Parámetros metodológicos
MIN_TWEETS_MACROTEMA = 10
MAX_MACROTEMAS = 6
MUESTRA_TWEETS_POR_TEMA = 30

# 1. Filtramos categorías que no aportan valor cualitativo
df_macrotemas_validos = df_corpus_final[
    ~df_corpus_final["macrotema_codigo"].isin(["NOINT", "OTROS"])
].copy()

# 2. Identificamos los macrotemas más relevantes por volumen
df_top_macrotemas = (
    df_macrotemas_validos.groupby(["macrotema_codigo", "macrotema"])
    .size()
    .reset_index(name="n_tweets")
    .sort_values("n_tweets", ascending=False)
    .head(MAX_MACROTEMAS)
    .reset_index(drop=True)
)

print(f"[+] Se analizarán los {len(df_top_macrotemas)} macrotemas principales:")
display(df_top_macrotemas)

# 3. Extraemos las muestras para cada macrotema seleccionado
muestras_por_tema = {}

for row in df_top_macrotemas.itertuples():
    if row.n_tweets >= MIN_TWEETS_MACROTEMA:
        muestra = df_macrotemas_validos[
            df_macrotemas_validos["macrotema_codigo"] == row.macrotema_codigo
        ]["tweet"].dropna().sample(
            n=min(row.n_tweets, MUESTRA_TWEETS_POR_TEMA),
            random_state=42 # Semilla fija para reproducibilidad académica
        ).tolist()

        muestras_por_tema[row.macrotema_codigo] = muestra

print(f"[+] Muestras textuales extraídas y listas para inferencia LLM.")

### 10.2 Ejecución del Análisis Cualitativo Asistido (LLM)



Para cada macrotema seleccionado, el Modelo de Lenguaje evalúa el contexto de la muestra representativa. Se aplica un *prompt* restrictivo que prohíbe inferencias clínicas, obligando al modelo a centrarse estrictamente en el análisis sociológico del discurso: focos de queja, objetos de atención y tono predominante.

In [ ]:
# ============================================================
# 10.2 EJECUCIÓN DEL ANÁLISIS CUALITATIVO (VÍA GEMINI)
# ============================================================

import json
import time

print(f"[*] Paso 10.2: Iniciando inferencia cualitativa de macrotemas...")

def analizar_macrotema_cualitativo(codigo, nombre, tweets_muestra):
    tweets_texto = "\n".join([f"- {t}" for t in tweets_muestra])

    prompt = f"""
Eres un investigador sociológico experto en análisis cualitativo de discurso.
Analiza este macrotema ("{nombre}") dentro de un corpus de usuarios de {COUNTRY} que han mostrado indicadores de riesgo psicológico previo.

Reglas metodológicas:
1. NO diagnostiques ni uses términos clínicos (es un análisis sociológico, no médico).
2. Analiza de qué hablan en su cotidianidad, qué les frustra o qué les motiva.
3. Devuelve estrictamente un objeto JSON válido con la estructura solicitada.

Estructura JSON requerida:
{{
  "resumen_interpretativo": "Resumen de 3 o 4 líneas sobre la articulación del discurso en este tema.",
  "actores_o_entidades": ["Actor 1", "Entidad 2"],
  "focos_de_preocupacion": ["Preocupación 1", "Frustración 2"],
  "tono_discursivo": "Descripción analítica del tono (ej. queja institucional, ironía, agotamiento).",
  "fragmento_representativo": "Extrae una cita textual corta de la muestra que resuma perfectamente el sentimiento general."
}}

Tweets de muestra:
{tweets_texto}
"""

    try:
        response = modelo_gemini.generate_content(prompt)
        texto = response.text.replace("```json", "").replace("```", "").strip()
        return json.loads(texto)
    except Exception as e:
        print(f"    [!] Error en la inferencia de {codigo}: {e}")
        return {}

resultados_cualitativos = []

for row in df_top_macrotemas.itertuples():
    codigo = row.macrotema_codigo

    if codigo not in muestras_por_tema:
        continue

    print(f"  -> Analizando constructo discursivo de: [{codigo}] {row.macrotema}...")

    analisis = analizar_macrotema_cualitativo(codigo, row.macrotema, muestras_por_tema[codigo])

    if analisis:
        resultados_cualitativos.append({
            "territorio": COUNTRY,
            "macrotema_codigo": codigo,
            "macrotema": row.macrotema,
            "volumen_tweets": row.n_tweets,
            "resumen_interpretativo": analisis.get("resumen_interpretativo", ""),
            "actores_o_entidades": " | ".join(analisis.get("actores_o_entidades", [])),
            "focos_de_preocupacion": " | ".join(analisis.get("focos_de_preocupacion", [])),
            "tono_discursivo": analisis.get("tono_discursivo", ""),
            "fragmento_representativo": analisis.get("fragmento_representativo", "")
        })

    # Latencia mínima (Pospago)
    time.sleep(2)

print("\n[+] Análisis cualitativo asistido finalizado.")

### 10.3 Validación y Exportación de la Matriz Cualitativa



Los resultados devueltos por el modelo se compilan en una tabla estructurada (Matriz Cualitativa). Este artefacto de datos unifica las métricas de volumen (cuantitativas) con las dimensiones de análisis discursivo (cualitativas), quedando listo para su integración en la memoria final de la investigación.

In [ ]:
# ============================================================
# 10.3 VALIDACIÓN, MATRIZ Y EXPORTACIÓN
# ============================================================

print(f"[*] Paso 10.3: Consolidando matriz cualitativa para {COUNTRY}...")

# 1. Conversión de los resultados a DataFrame
df_matriz_cualitativa = pd.DataFrame(resultados_cualitativos)

if df_matriz_cualitativa.empty:
    print("[!] Advertencia: La matriz cualitativa está vacía. Revisa la ejecución del Paso 10.2.")
else:
    # 2. Definición de rutas dinámicas de salida
    ruta_matriz = f"{OUTPUT_DIR}/10_{COUNTRY_CODE}_matriz_cualitativa_integrada.csv"

    # 3. Exportación
    df_matriz_cualitativa.to_csv(ruta_matriz, index=False, encoding="utf-8-sig")

    print(f"[+] Archivo CSV exportado correctamente en:\n    - {ruta_matriz}\n")

    # 4. Auditoría visual de la Matriz Final
    print("=" * 80)
    print("VISTA PREVIA DE LA MATRIZ CUALITATIVA")
    print("=" * 80)
    display(df_matriz_cualitativa)

## 11. Control Final de Calidad y Auditoría de Exportación



Este bloque final actúa como una herramienta de validación del *Pipeline* de Datos. Su objetivo es realizar una auditoría automatizada del estado de las variables en memoria y verificar la correcta exportación física de todos los artefactos de datos (archivos CSV y JSON). Esta comprobación garantiza la integridad metodológica antes de proceder con el análisis de un nuevo territorio.

In [ ]:
# ============================================================
# 11. AUDITORÍA FINAL DEL PIPELINE
# ============================================================

import os

print("=" * 50)
print("AUDITORÍA DE CALIDAD DEL PIPELINE")
print("=" * 50)

print(f"Territorio analizado: {COUNTRY}")
print(f"Código identificador: {COUNTRY_CODE}")
print(f"Carpeta de salida: {OUTPUT_DIR}\n")

# --- CONTROL DE MEMORIA (DATA FRAMES) ---
print("--- MÉTRICAS DE RETENCIÓN DE MUESTRA ---")
print(f"Paso 1 (Semilla API): {len(df_semilla) if 'df_semilla' in globals() else 'No disponible'} usuarios")
print(f"Paso 3 (Físicos): {len(df_usuarios_finales) if 'df_usuarios_finales' in globals() else 'No disponible'} usuarios")
print(f"Paso 7 (Clasificados): {len(df_usuarios) if 'df_usuarios' in globals() else 'No disponible'} usuarios")

if "df_usuarios" in globals() and "clasificacion_usuario" in df_usuarios.columns:
    seleccionados = (df_usuarios["clasificacion_usuario"] == "seleccionado").sum()
    control = (df_usuarios["clasificacion_usuario"] == "control").sum()
    print(f"  -> Grupo Riesgo: {seleccionados} | Grupo Control: {control}")

print(f"Paso 7 (Tweets Cotidianos): {len(df_cotidiano) if 'df_cotidiano' in globals() else 'No disponible'} publicaciones")
print(f"Paso 8 (Microtemas): {len(df_tweets_microtemas) if 'df_tweets_microtemas' in globals() else 'No disponible'} clasificaciones")
print(f"Paso 9 (Macrotemas): {len(df_corpus_final) if 'df_corpus_final' in globals() else 'No disponible'} normalizaciones")
print(f"Paso 10 (Matriz Cualitativa): {len(df_matriz_cualitativa) if 'df_matriz_cualitativa' in globals() else 'No disponible'} macrotemas analizados")

# --- CONTROL DE EXPORTACIÓN (ARCHIVOS FÍSICOS) ---
print("\n--- ESTADO DE EXPORTACIÓN DE ARTEFACTOS ---")

archivos_esperados = [
    f"01_{COUNTRY_CODE}_dataset_usuarios_semilla_V3.csv",
    f"02_{COUNTRY_CODE}_dataset_usuarios_semilla_clasificados.csv",
    f"03_{COUNTRY_CODE}_dataset_usuarios_fisicos.csv",
    f"04_{COUNTRY_CODE}_dataset_contexto_historico.json",
    f"05_{COUNTRY_CODE}_dataset_anonimizado.json",
    f"06_{COUNTRY_CODE}_dataset_clasificado_salud_mental.json",
    f"06_{COUNTRY_CODE}_resumen_clasificacion_salud_mental.csv",
    f"07_{COUNTRY_CODE}_usuarios_clasificados_seleccionados_control.csv",
    f"07_{COUNTRY_CODE}_corpus_cotidiano_usuarios_seleccionados.csv",
    f"08_{COUNTRY_CODE}_tweets_cotidianos_microtemas.csv",
    f"08_{COUNTRY_CODE}_frecuencia_microtemas.csv",
    f"09_{COUNTRY_CODE}_corpus_tematico_final.csv",
    f"09_{COUNTRY_CODE}_resumen_distribucion_macrotemas.csv",
    f"10_{COUNTRY_CODE}_matriz_cualitativa_integrada.csv"
]

archivos_faltantes = 0

for archivo in archivos_esperados:
    ruta = os.path.join(OUTPUT_DIR, archivo)
    if os.path.exists(ruta):
        print(f"[✔] OK - {archivo}")
    else:
        print(f"[!] FALTANTE - {archivo}")
        archivos_faltantes += 1

print("-" * 50)
if archivos_faltantes == 0:
    print("✅ ESTADO FINAL: ÉXITO. Todos los artefactos han sido generados correctamente.")
    print("Puedes cambiar la variable COUNTRY al inicio del Colab y proceder con el siguiente país.")
else:
    print(f"⚠️ ESTADO FINAL: ALERTA. Faltan {archivos_faltantes} archivos. Revisa las celdas anteriores.")
print("=" * 50)

In [ ]:
# =========================================================
# PASO 6.1. CARGA Y FUNCIÓN DE CLASIFICACIÓN SEMÁNTICA
# =========================================================

import json
import time
import pandas as pd
import google.generativeai as genai

# =========================================================
# CARGA DEL DATASET ANONIMIZADO (Ruta dinámica)
# =========================================================

ruta_archivo = f"{OUTPUT_DIR}/05_{COUNTRY_CODE}_dataset_anonimizado.json"

with open(ruta_archivo, "r", encoding="utf-8") as f:
    dataset_anonimizado = json.load(f)

print(
    f"[*] Paso 6.1: Datos cargados para {COUNTRY}. "
    f"Usuarios a analizar: {len(dataset_anonimizado)}"
)

# =========================================================
# CONFIGURACIÓN DE GEMINI
# =========================================================

genai.configure(api_key=GEMINI_API_KEY, transport="rest")
modelo_gemini = genai.GenerativeModel("gemini-2.5-flash")

# =========================================================
# FUNCIÓN DE CLASIFICACIÓN SEMÁNTICA
# =========================================================

def clasificar_tweets_salud_mental(lista_tweets, max_reintentos=3):
    """
    Clasificador semántico con sistema antibloqueo y manejo de cuotas.

    1 = Contenido relacionado con depresión, suicidio, autolesión o desesperanza.
    0 = Contenido no relacionado, coloquial, humorístico, figurado, informativo o ambiguo.
    """
    if not lista_tweets:
        return []

    tweets_formateados = "\n".join(
        [f"{i + 1}. {tweet}" for i, tweet in enumerate(lista_tweets)]
    )

    prompt = f"""
Actúa como experta en análisis semántico de discurso en redes sociales.

Vas a recibir una lista de tweets anonimizados de un mismo usuario.
Tu tarea es clasificar CADA tweet individualmente.

Criterio:

Etiqueta con 1 si el tweet expresa o menciona de forma clara alguno de estos elementos:
- depresión
- desesperanza
- deseo de morir
- ideación suicida
- intento suicida
- autolesión
- deseo de desaparecer
- sensación de vacío extremo
- sufrimiento emocional intenso asociado a no querer seguir viviendo

Etiqueta con 0 si:
- no está relacionado con depresión, suicidio o autolesión
- menciona salud mental de forma genérica, institucional, divulgativa o informativa
- el uso es coloquial, humorístico, metafórico o ambiguo
- habla de ansiedad, estrés o malestar general sin relación clara con depresión o suicidio
- no hay suficiente evidencia textual

Importante:
- No hagas diagnóstico clínico.
- Evalúa solo el contenido textual del tweet.
- No infieras información que no esté explícita o semánticamente clara.
- Responde SOLO en formato JSON válido.
- Devuelve exactamente una etiqueta por tweet.

Formato requerido:
[
  {{"tweet_id": 1, "label": 0}},
  {{"tweet_id": 2, "label": 1}}
]

Tweets:
{tweets_formateados}
"""

    for intento in range(max_reintentos):
        try:

            response = modelo_gemini.generate_content(prompt)

            texto = (
                response.text
                .replace("```json", "")
                .replace("```", "")
                .strip()
            )

            resultado = json.loads(texto)

            etiquetas = [
                item["label"]
                for item in resultado
            ]

            if len(etiquetas) != len(lista_tweets):
                print(
                    "    [!] Número de etiquetas inconsistente. "
                    "Se devolverán 0 por seguridad."
                )
                return [0] * len(lista_tweets)

            return [
                1 if x == 1 else 0
                for x in etiquetas
            ]

        except Exception as e:
            error_str = str(e)

            if "429" in error_str or "Quota" in error_str:
                print(
                    f"    [!] Límite de cuota alcanzado. "
                    f"Pausando 60 segundos "
                    f"(Intento {intento + 1}/{max_reintentos})..."
                )
                time.sleep(60)
            else:
                print(f"    [!] Error en clasificación LLM: {e}")
                return [0] * len(lista_tweets)

    return [0] * len(lista_tweets)

In [ ]:
# =========================================================
# PASO 6.2. CLASIFICACIÓN SEMÁNTICA DEL DATASET
# =========================================================

print(f"[*] Paso 6.2: Iniciando clasificación semántica del dataset de {COUNTRY}...")

dataset_clasificado = {}
total_usuarios = len(dataset_anonimizado)

for idx, (usuario, tweets) in enumerate(
    dataset_anonimizado.items(),
    start=1
):
    if idx % 10 == 0 or idx == 1:
        print(f"  -> Procesados {idx}/{total_usuarios} usuarios...")

    etiquetas = clasificar_tweets_salud_mental(tweets)

    dataset_clasificado[usuario] = [
        {
            "tweet": tweet,
            "mental_health": etiqueta
        }
        for tweet, etiqueta in zip(tweets, etiquetas)
    ]

    # Ventana de latencia de seguridad obligatoria para el Rate Limiting de Gemini
    time.sleep(6)

print("\n[+] Clasificación completada exitosamente.")

[*] Paso 6.2: Iniciando clasificación semántica del dataset de Euskera...
  -> Procesados 1/287 usuarios...
  -> Procesados 10/287 usuarios...
  -> Procesados 20/287 usuarios...
    [!] Error en clasificación LLM: HTTPConnectionPool(host='localhost', port=34587): Read timed out. (read timeout=600.0)
    [!] Error en clasificación LLM: HTTPConnectionPool(host='localhost', port=34587): Read timed out. (read timeout=600.0)
    [!] Error en clasificación LLM: HTTPConnectionPool(host='localhost', port=34587): Read timed out. (read timeout=600.0)
    [!] Error en clasificación LLM: HTTPConnectionPool(host='localhost', port=34587): Read timed out. (read timeout=600.0)
  -> Procesados 30/287 usuarios...
    [!] Error en clasificación LLM: HTTPConnectionPool(host='localhost', port=34587): Read timed out. (read timeout=600.0)
    [!] Error en clasificación LLM: HTTPConnectionPool(host='localhost', port=34587): Read timed out. (read timeout=600.0)
    [!] Error en clasificación LLM: HTTPConnectio